# CT032-3-3 Further Artificial Intelligence (FAI)
## Part 2: AI Software Artifact â€” Walmart Weekly Sales Forecasting

**Group H | APD3F2511CS(AI)**

| Role | Student ID & Name |
|------|-------------------|
| Group Leader | SANJIVAN A/L R.T. THIYAGESWARAN (TP070073) |
| Group Member 1 | DEVARA ALANDRA WICAKSONO (TP073570) |
| Group Member 2 | JIYAD MOHAMMED ISMAIL MAYYAS (TP077380) |
| Group Member 3 | TANESHEN A/L MAHINDRAN (TP078396) |
| Group Member 4 | FAREEZ DANIEL BIN ROZAIME (TP077930) |

**Lecturer:** Dr. Adeline Sneha John Chrisastum

---

## Overview

This notebook implements the machine learning solution designed in Part 1 for predicting weekly retail sales across Walmart stores and departments. The system follows the architecture pipeline defined in Part 1:

1. **Data Ingestion** â€” Load and merge `train.csv`, `features.csv`, and `stores.csv`
2. **Data Preprocessing** â€” Handle missing values, encode categoricals
3. **Feature Engineering** â€” Extract temporal features from dates
4. **Feature Selection** â€” Scenario A (leakage-safe, no MarkDowns) as primary
5. **Model Training** â€” XGBoost (primary) + Random Forest (validation layer)
6. **Evaluation** â€” MAE, RMSE, MAPE with holiday vs non-holiday breakdown
7. **Visualization** â€” Actual vs Predicted, feature importance, SHAP analysis
8. **Scenario B** â€” Secondary comparison including MarkDown variables

---
## Section 0: Environment Setup

This cell automatically detects whether the notebook is running on **Google Colab** or a **local machine** and adjusts accordingly:
- On Colab: installs missing packages and prompts you to upload the 3 dataset files
- On local: assumes packages are already installed and uses the `FAI Dataset/` folder

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 0: Environment Setup (Colab + Local compatible)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

import sys
import os

# Detect if running on Google Colab
ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    print("Google Colab environment detected.")

    # Install packages not pre-installed on Colab
    print("Installing required packages...")
    os.system('pip install -q xgboost shap')
    print("Packages ready.")

    # Upload dataset files
    print("\nPlease upload the 3 dataset files: train.csv, features.csv, stores.csv")
    from google.colab import files
    uploaded = files.upload()  # Opens file picker â€” select all 3 CSVs

    # Files are uploaded to /content/ on Colab
    DATA_PATH = '/content/'
    print(f"\nFiles uploaded: {list(uploaded.keys())}")

else:
    print("Local environment detected.")
    # Dataset folder relative to this notebook
    DATA_PATH = 'FAI Dataset/'

print(f"Data path set to: '{DATA_PATH}'")

---
## Section 1: Library Imports

All required libraries for data processing, machine learning, and visualization are imported here.

**Libraries used:**
- `pandas`, `numpy` â€” Data manipulation and numerical computation
- `matplotlib`, `seaborn` â€” Data visualization
- `scikit-learn` â€” Random Forest, preprocessing, and evaluation metrics
- `xgboost` â€” Primary prediction algorithm (XGBoost Regressor)
- `shap` â€” Model explainability (SHapley Additive exPlanations)

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 1: Library Imports
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

import xgboost as xgb

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed â€” skipping explainability section.")

warnings.filterwarnings('ignore')

# Consistent plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('muted')

print("All libraries loaded successfully.")
print(f"XGBoost version  : {xgb.__version__}")
print(f"Pandas version   : {pd.__version__}")
print(f"NumPy version    : {np.__version__}")
print(f"Running on Colab : {ON_COLAB}")

---
## Section 2: Data Ingestion Layer

Following the architecture designed in Part 1 (Section 2.1.1), this layer loads and integrates the three raw data sources:

| File | Contents |
|------|----------|
| `train.csv` | Historical weekly sales per Store/Dept |
| `features.csv` | External indicators (temperature, fuel price, CPI, etc.) |
| `stores.csv` | Store-level structural attributes (Type, Size) |

The datasets are merged on `Store` and `Date` to form a single consolidated dataframe.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 2: Data Ingestion Layer
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Load raw datasets from the detected data path
train    = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
features = pd.read_csv(os.path.join(DATA_PATH, 'features.csv'))
stores   = pd.read_csv(os.path.join(DATA_PATH, 'stores.csv'))

print("Dataset shapes:")
print(f"  train.csv    : {train.shape}")
print(f"  features.csv : {features.shape}")
print(f"  stores.csv   : {stores.shape}")

# Step 1: Merge train with features on Store + Date
# IsHoliday exists in both; '_feat' suffix added to features version to avoid collision
df = train.merge(
    features,
    on=['Store', 'Date'],
    how='left',
    suffixes=('', '_feat')
)

# Step 2: Merge with stores on Store
df = df.merge(stores, on='Store', how='left')

print(f"\nMerged dataframe shape : {df.shape}")
print(f"Date range             : {df['Date'].min()} to {df['Date'].max()}")
print(f"Unique stores          : {df['Store'].nunique()}")
print(f"Unique departments     : {df['Dept'].nunique()}")

---
## Section 3: Data Preprocessing

Following Part 1 (Section 2.1.2), preprocessing includes:

1. **IsHoliday consistency check** â€” Verify both `IsHoliday` columns match after merging; retain one
2. **Date parsing** â€” Convert `Date` column to datetime format
3. **Categorical encoding** â€” Label-encode `Type` (A/B/C â†’ 0/1/2)
4. **Missing value inspection** â€” Check missingness per feature
5. **Mean imputation** â€” Applied only to primary feature set numeric columns

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 3: Data Preprocessing
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# 3.1 IsHoliday Consistency Check
# 'IsHoliday' from train.csv vs 'IsHoliday_feat' from features.csv
if 'IsHoliday_feat' in df.columns:
    mismatch = (df['IsHoliday'] != df['IsHoliday_feat']).sum()
    print(f"IsHoliday consistency check â€” mismatched rows: {mismatch}")
    df.drop(columns=['IsHoliday_feat'], inplace=True)

# 3.2 Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# 3.3 Encode IsHoliday as integer (True/False â†’ 1/0)
df['IsHoliday'] = df['IsHoliday'].astype(int)

# 3.4 Label-encode Store Type (A, B, C â†’ 0, 1, 2)
le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])
print(f"Type encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 3.5 Missingness report
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("\nMissing value % per column:")
print(missing_pct[missing_pct > 0].round(2).to_string())

# 3.6 Mean imputation for primary feature numeric columns only
for col in ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

print("\nPreprocessing complete.")
print(f"Final shape: {df.shape}")

---
## Section 4: Feature Engineering

Following Part 1 (Section 2.1.3), temporal features are extracted from the `Date` column:

| Feature | Rationale |
|---------|-----------|
| `year` | Captures year-level trends |
| `month` | Captures monthly seasonal cycles |
| `week` | Week-of-year; captures fine-grained seasonality and holiday proximity |

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 4: Feature Engineering
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Extract temporal features from Date
df['year']  = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['week']  = df['Date'].dt.isocalendar().week.astype(int)

# Sort chronologically by Store, Dept, Date
df.sort_values(['Store', 'Dept', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Temporal features extracted: year, month, week")
print(f"Year range  : {df['year'].min()} â€“ {df['year'].max()}")
print(f"Month range : {df['month'].min()} â€“ {df['month'].max()}")
print(f"Week range  : {df['week'].min()} â€“ {df['week'].max()}")
df[['Store','Dept','Date','year','month','week','IsHoliday','Weekly_Sales']].head(3)

---
## Section 5: Exploratory Data Analysis (EDA)

Visual evidence supporting the feature selection decisions made in Part 1:
- Weekly sales trend over time (justifies temporal features)
- Holiday vs non-holiday effect (justifies `IsHoliday`)
- Missingness per feature (justifies MarkDown exclusion from Scenario A)
- Sales distribution by store type (justifies `Type` and `Size`)

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 5: Exploratory Data Analysis
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Exploratory Data Analysis â€” Walmart Weekly Sales', fontsize=14, fontweight='bold')

# Plot 1: Mean Weekly Sales Over Time
ax1 = axes[0, 0]
weekly_mean = df.groupby('Date')['Weekly_Sales'].mean()
ax1.plot(weekly_mean.index, weekly_mean.values, color='steelblue', linewidth=1.2)
ax1.set_title('Mean Weekly Sales Over Time')
ax1.set_xlabel('Date')
ax1.set_ylabel('Mean Weekly Sales ($)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Plot 2: Holiday vs Non-Holiday Sales
ax2 = axes[0, 1]
holiday_mean = df.groupby('IsHoliday')['Weekly_Sales'].mean()
bars = ax2.bar(['Non-Holiday', 'Holiday'], holiday_mean.values,
               color=['#4c9be8', '#e87b4c'], edgecolor='white')
ax2.set_title('Holiday vs Non-Holiday Effect (Mean Weekly Sales)')
ax2.set_ylabel('Mean Weekly Sales ($)')
for bar, val in zip(bars, holiday_mean.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'${val:,.0f}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Missingness Per Feature
ax3 = axes[1, 0]
all_feat_cols = ['Store','Dept','Type','Size','IsHoliday','Temperature','Fuel_Price',
                 'CPI','Unemployment','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
miss_pct = [(df[c].isnull().sum() / len(df) * 100) if c in df.columns else 0 for c in all_feat_cols]
colors_miss = ['#e05c5c' if m > 10 else '#4cae4c' for m in miss_pct]
ax3.barh(all_feat_cols, miss_pct, color=colors_miss)
ax3.set_title('Missingness (%) per Feature')
ax3.set_xlabel('% Missing')
ax3.axvline(10, color='red', linestyle='--', alpha=0.6, label='10% threshold')
ax3.legend()

# Plot 4: Sales Distribution by Store Type
ax4 = axes[1, 1]
type_labels = {v: k for k, v in dict(zip(le.classes_, le.transform(le.classes_))).items()}
for t_val, t_name in type_labels.items():
    subset = df[df['Type'] == t_val]['Weekly_Sales']
    ax4.hist(subset, bins=60, alpha=0.6, label=f'Type {t_name}', density=True)
ax4.set_title('Weekly Sales Distribution by Store Type')
ax4.set_xlabel('Weekly Sales ($)')
ax4.set_ylabel('Density')
ax4.set_xlim(-5000, 150000)
ax4.legend()

plt.tight_layout()
plt.savefig('plot_01_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_01_eda.png")

---
## Section 6: Feature Selection â€” Scenario A (Primary)

Based on Part 1 (Section 1.6.6), **Scenario A** is the primary feature set â€” excluding MarkDown variables due to >60% missingness.

| Feature Group | Variables |
|---------------|-----------|
| Identifiers | Store, Dept |
| Temporal | year, month, week |
| Event | IsHoliday |
| Store structure | Type, Size |
| Economic/External | Temperature, Fuel_Price, CPI, Unemployment |

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 6: Feature Selection
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

TARGET = 'Weekly_Sales'

# Scenario A: leakage-safe, no MarkDowns
FEATURES_A = [
    'Store', 'Dept',
    'year', 'month', 'week',
    'IsHoliday',
    'Type', 'Size',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'
]

# Scenario B: MarkDowns included with zero imputation + missingness flags
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df_b = df.copy()
for col in markdown_cols:
    if col in df_b.columns:
        df_b[f'{col}_missing'] = df_b[col].isnull().astype(int)
        df_b[col].fillna(0, inplace=True)

FEATURES_B = FEATURES_A + markdown_cols + [f'{c}_missing' for c in markdown_cols]

print(f"Scenario A â€” {len(FEATURES_A)} features: {FEATURES_A}")
print(f"\nScenario B â€” {len(FEATURES_B)} features: {FEATURES_B}")

---
## Section 7: Time-Based Train/Test Split

A **chronological split** is used (Part 1, Section 3.4) â€” last 20% of unique weeks as test set, preventing data leakage.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 7: Chronological Train/Test Split
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

unique_dates = sorted(df['Date'].unique())
n_dates      = len(unique_dates)
split_idx    = int(n_dates * 0.80)
cutoff_date  = unique_dates[split_idx]

train_mask = df['Date'] < cutoff_date
test_mask  = df['Date'] >= cutoff_date

print(f"Total unique weeks : {n_dates}")
print(f"Cutoff date        : {pd.Timestamp(cutoff_date).date()}")
print(f"Train rows         : {train_mask.sum():,}")
print(f"Test rows          : {test_mask.sum():,}")

X_train_A = df.loc[train_mask, FEATURES_A]
y_train   = df.loc[train_mask, TARGET]
X_test_A  = df.loc[test_mask,  FEATURES_A]
y_test    = df.loc[test_mask,  TARGET]

X_train_B = df_b.loc[train_mask, FEATURES_B]
X_test_B  = df_b.loc[test_mask,  FEATURES_B]

test_dates   = df.loc[test_mask, 'Date']
test_holiday = df.loc[test_mask, 'IsHoliday']

print("\nSplit complete.")

---
## Section 8: Model Training

### 8.1 Primary Model â€” XGBoost Regressor

XGBoost is selected as the primary algorithm (Part 1, Section 3.3.1) due to sparsity-awareness, L1/L2 regularization, automatic non-linear interaction capture, and proven performance in the M5 Forecasting Competition.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `n_estimators` | 500 | Sufficient boosting rounds for dataset size |
| `learning_rate` | 0.05 | Low rate + more trees reduces overfitting |
| `max_depth` | 6 | Captures higher-order feature interactions |
| `subsample` | 0.8 | Row subsampling for variance reduction |
| `colsample_bytree` | 0.8 | Column subsampling per tree |
| `reg_lambda` | 1.5 | L2 regularization on leaf weights |
| `reg_alpha` | 0.1 | L1 regularization for feature sparsity |
| `min_child_weight` | 5 | Prevents overfitting to small groups |

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 8.1: XGBoost Regressor â€” Scenario A (Primary)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

xgb_model = xgb.XGBRegressor(
    n_estimators     = 500,
    learning_rate    = 0.05,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_lambda       = 1.5,
    reg_alpha        = 0.1,
    min_child_weight = 5,
    objective        = 'reg:squarederror',
    random_state     = 42,
    n_jobs           = -1,
    verbosity        = 0
)

print("Training XGBoost Regressor (Scenario A)...")
xgb_model.fit(X_train_A, y_train)

# Clip negative predictions to 0 (post-processing step from Part 1, Section 3.4)
y_pred_xgb_A = np.clip(xgb_model.predict(X_test_A), 0, None)
print("Done.")

### 8.2 Secondary Model â€” Random Forest (Validation Layer)

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 8.2: Random Forest â€” Validation Layer (Part 1, Section 3.3.2)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

rf_model = RandomForestRegressor(
    n_estimators     = 200,
    max_depth        = 12,
    max_features     = 0.6,
    min_samples_leaf = 4,
    random_state     = 42,
    n_jobs           = -1
)

print("Training Random Forest (validation layer)...")
rf_model.fit(X_train_A, y_train)
y_pred_rf_A = np.clip(rf_model.predict(X_test_A), 0, None)
print("Done.")

### 8.3 XGBoost â€” Scenario B (MarkDowns Included)

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 8.3: XGBoost â€” Scenario B
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

xgb_model_B = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5,
    reg_alpha=0.1, min_child_weight=5,
    objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0
)

print("Training XGBoost Regressor (Scenario B â€” with MarkDowns)...")
xgb_model_B.fit(X_train_B, y_train)
y_pred_xgb_B = np.clip(xgb_model_B.predict(X_test_B), 0, None)
print("Done.")

---
## Section 9: Model Evaluation

| Metric | Interpretation |
|--------|----------------|
| MAE | Average absolute dollar error per prediction |
| RMSE | Penalises large errors more heavily than MAE |
| WMAPE | Weighted MAPE â€” scale-robust percentage error; avoids inflation from low-volume rows |

> **Note on WMAPE vs MAPE:** Standard MAPE divides each error by the actual value, which blows up when actual sales are near zero. Approximately 6.4% of rows in this dataset have Weekly_Sales < $100. WMAPE = Î£\|actual âˆ’ pred\| / Î£actual Ã— 100 avoids this by weighting errors by their share of total sales volume, making it the appropriate percentage metric for this dataset.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 9: Model Evaluation
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def compute_metrics(y_true, y_pred, label=''):
    """Compute MAE, RMSE, and WMAPE (Weighted Mean Absolute Percentage Error).

    Standard MAPE is unstable when actual sales are near zero â€” dividing by a
    small denominator inflates the metric arbitrarily. This dataset contains
    ~6.4% of rows with Weekly_Sales < $100 (low-volume departments), which
    would produce misleadingly large MAPE values.

    WMAPE = sum(|actual - pred|) / sum(actual) * 100
    is scale-weighted so low-sales rows do not disproportionately affect the
    result. Only rows with actual sales > 0 are included in the denominator.
    """
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mask  = y_true > 0
    wmape = np.sum(np.abs(y_true[mask] - y_pred[mask])) / np.sum(y_true[mask]) * 100
    if label:
        print(f"  {label}")
        print(f"    MAE   : ${mae:>12,.2f}")
        print(f"    RMSE  : ${rmse:>12,.2f}")
        print(f"    WMAPE : {wmape:>10.2f}%")
    return mae, rmse, wmape

y_test_arr = y_test.values

print("=" * 55)
print("Overall Performance on Test Set")
print("=" * 55)
mae_xgb_A, rmse_xgb_A, wmape_xgb_A = compute_metrics(y_test_arr, y_pred_xgb_A, "XGBoost â€” Scenario A (Primary)")
print()
mae_rf_A,  rmse_rf_A,  wmape_rf_A  = compute_metrics(y_test_arr, y_pred_rf_A,  "Random Forest â€” Scenario A (Validation)")
print()
mae_xgb_B, rmse_xgb_B, wmape_xgb_B = compute_metrics(y_test_arr, y_pred_xgb_B, "XGBoost â€” Scenario B (MarkDowns)")

print("\n" + "=" * 55)
print("Scenario Comparison")
print("=" * 55)
print(pd.DataFrame({
    'Scenario':   ['A â€” No MarkDowns', 'B â€” MarkDowns (zero impute)'],
    'MAE':        [round(mae_xgb_A, 2),   round(mae_xgb_B, 2)],
    'RMSE':       [round(rmse_xgb_A, 2),  round(rmse_xgb_B, 2)],
    'WMAPE (%)':  [round(wmape_xgb_A, 2), round(wmape_xgb_B, 2)]
}).to_string(index=False))

print("\n" + "=" * 55)
print("Holiday vs Non-Holiday Breakdown (XGBoost Scenario A)")
print("=" * 55)
hol_idx     = test_holiday.values == 1
non_hol_idx = test_holiday.values == 0
if hol_idx.sum() > 0:
    compute_metrics(y_test_arr[hol_idx],     y_pred_xgb_A[hol_idx],     "Holiday weeks")
    print()
    compute_metrics(y_test_arr[non_hol_idx], y_pred_xgb_A[non_hol_idx], "Non-Holiday weeks")

---
## Section 10: Data Visualization and Results Presentation

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 10.1: Actual vs Predicted â€” Mean Weekly Sales
#
# Caption: Mean actual vs predicted weekly sales over the test period.
# Red bands highlight holiday weeks where demand spikes are observed.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

test_df_plot = pd.DataFrame({
    'Date': test_dates.values, 'Actual': y_test_arr,
    'Predicted': y_pred_xgb_A, 'IsHoliday': test_holiday.values
})
weekly_actual = test_df_plot.groupby('Date')['Actual'].mean()
weekly_pred   = test_df_plot.groupby('Date')['Predicted'].mean()
weekly_hol    = test_df_plot.groupby('Date')['IsHoliday'].max()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly_actual.index, weekly_actual.values, label='Actual', color='steelblue', linewidth=1.5)
ax.plot(weekly_pred.index, weekly_pred.values, label='XGBoost Predicted',
        color='orange', linewidth=1.5, linestyle='--')
for date, is_hol in weekly_hol.items():
    if is_hol:
        ax.axvline(date, color='red', alpha=0.15, linewidth=6)
ax.set_title('Actual vs Predicted Weekly Sales (Test Set)\nRed bands = Holiday weeks', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Mean Weekly Sales ($)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plot_02_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_02_actual_vs_predicted.png")

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 10.2: Scatter Plot â€” Actual vs Predicted
#
# Caption: Each point is one Store/Dept/week prediction. Points on the
# red diagonal indicate perfect predictions.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

sample_idx = np.random.choice(len(y_test_arr), size=min(5000, len(y_test_arr)), replace=False)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test_arr[sample_idx], y_pred_xgb_A[sample_idx], alpha=0.25, s=8, color='steelblue')
max_val = max(y_test_arr[sample_idx].max(), y_pred_xgb_A[sample_idx].max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_title(f'Actual vs Predicted Scatter (n=5,000 sample)\nMAE = ${mae_xgb_A:,.0f}')
ax.set_xlabel('Actual Weekly Sales ($)')
ax.set_ylabel('Predicted Weekly Sales ($)')
ax.legend()
plt.tight_layout()
plt.savefig('plot_03_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_03_scatter.png")

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 10.3: Feature Importance Comparison
#
# Caption: XGBoost gain-based importance (left) vs Random Forest Gini
# importance (right). Both consistently rank Dept and Size highest,
# validating the Part 1 feature selection.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

xgb_imp = pd.Series(
    xgb_model.get_booster().get_score(importance_type='gain'),
    name='XGBoost Gain'
).reindex(FEATURES_A).fillna(0).sort_values(ascending=True)

rf_imp = pd.Series(
    rf_model.feature_importances_, index=FEATURES_A
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Comparison', fontsize=13, fontweight='bold')
axes[0].barh(xgb_imp.index, xgb_imp.values, color='steelblue')
axes[0].set_title('XGBoost â€” Feature Importance (Gain)')
axes[0].set_xlabel('Gain Score')
axes[1].barh(rf_imp.index, rf_imp.values, color='darkorange')
axes[1].set_title('Random Forest â€” Feature Importance (Gini)')
axes[1].set_xlabel('Gini Score')
plt.tight_layout()
plt.savefig('plot_04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_04_feature_importance.png")

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 10.4: Model Performance Comparison
#
# Caption: MAE and RMSE across all three models. XGBoost Scenario A
# achieves the best balance of accuracy and forecasting realism.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

models      = ['XGBoost\nScenario A', 'Random Forest\nScenario A', 'XGBoost\nScenario B']
mae_vals    = [mae_xgb_A,  mae_rf_A,  mae_xgb_B]
rmse_vals   = [rmse_xgb_A, rmse_rf_A, rmse_xgb_B]
x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, mae_vals,  width, label='MAE',  color='steelblue',  alpha=0.85)
b2 = ax.bar(x + width/2, rmse_vals, width, label='RMSE', color='darkorange', alpha=0.85)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Error ($)')
ax.set_title('Model Performance Comparison â€” MAE & RMSE')
ax.legend()
plt.tight_layout()
plt.savefig('plot_05_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_05_model_comparison.png")

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 10.5: Residual Distribution
#
# Caption: Residuals centred near zero indicate an unbiased model. Slight
# positive skew reflects under-prediction during holiday sales spikes.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

residuals = y_test_arr - y_pred_xgb_A
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=80, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
axes[0].axvline(residuals.mean(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Mean = ${residuals.mean():,.0f}')
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residual ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

wr = test_df_plot.copy()
wr['Residual'] = residuals
wr = wr.groupby('Date')['Residual'].mean()
axes[1].plot(wr.index, wr.values, color='steelblue', linewidth=1)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Mean Residuals Over Time')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Mean Residual ($)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plot_06_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_06_residuals.png")

---
## Section 11: SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) values address the 'black box' limitation of ensemble models (Part 1, Section 3.6). Each feature receives a contribution score per prediction.

**Caption:** Beeswarm plot â€” features ranked by mean |SHAP value|. Red = high feature value, Blue = low. For example, high `Dept` values drive large positive or negative deviations from the base prediction depending on the department's baseline sales level.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 11: SHAP Explainability
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

if SHAP_AVAILABLE:
    print("Computing SHAP values (sampling 2,000 rows for speed)...")
    shap_idx    = np.random.choice(len(X_test_A), size=min(2000, len(X_test_A)), replace=False)
    X_shap      = X_test_A.iloc[shap_idx]
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_shap)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_shap, feature_names=FEATURES_A,
                      plot_type='dot', show=False, max_display=12)
    plt.title('SHAP Beeswarm â€” XGBoost Scenario A', fontsize=12)
    plt.tight_layout()
    plt.savefig('plot_07_shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: plot_07_shap_beeswarm.png")

    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values, X_shap, feature_names=FEATURES_A,
                      plot_type='bar', show=False, max_display=12)
    plt.title('Mean |SHAP Value| â€” Feature Importance', fontsize=12)
    plt.tight_layout()
    plt.savefig('plot_08_shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: plot_08_shap_bar.png")
else:
    print("SHAP not available. Skipping.")

---
## Section 12: Demonstration â€” Store 1, Department 1

**Caption:** Full sales timeline for Store 1, Dept 1. Blue = training period, green = test actual, orange dashed = XGBoost predictions. Red bands mark holiday weeks. The vertical dotted line is the train/test split boundary.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 12: Demonstration â€” Store 1, Department 1
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

DEMO_STORE, DEMO_DEPT = 1, 1

demo_train = df[(df['Store'] == DEMO_STORE) & (df['Dept'] == DEMO_DEPT) & train_mask]
demo_test  = df[(df['Store'] == DEMO_STORE) & (df['Dept'] == DEMO_DEPT) & test_mask]
demo_pred  = np.clip(xgb_model.predict(demo_test[FEATURES_A]), 0, None)
demo_actual = demo_test['Weekly_Sales'].values

demo_mae  = mean_absolute_error(demo_actual, demo_pred)
demo_rmse = np.sqrt(mean_squared_error(demo_actual, demo_pred))
print(f"Store {DEMO_STORE}, Dept {DEMO_DEPT} â€” Test MAE: ${demo_mae:,.2f} | RMSE: ${demo_rmse:,.2f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(demo_train['Date'], demo_train['Weekly_Sales'], color='steelblue', linewidth=1.2, label='Training Actual')
ax.plot(demo_test['Date'],  demo_actual, color='green',  linewidth=1.5, label='Test Actual')
ax.plot(demo_test['Date'],  demo_pred,   color='orange', linewidth=1.5, linestyle='--', label='XGBoost Predicted')
for hd in demo_test.loc[demo_test['IsHoliday'] == 1, 'Date']:
    ax.axvline(hd, color='red', alpha=0.3, linewidth=5)
ax.axvline(pd.Timestamp(cutoff_date), color='black', linestyle=':', linewidth=1.5, label='Train/Test split')
ax.set_title(f'Store {DEMO_STORE} â€” Dept {DEMO_DEPT}: Actual vs Predicted\nRed = Holiday | MAE=${demo_mae:,.0f}', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Sales ($)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plot_09_demo_store1_dept1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: plot_09_demo_store1_dept1.png")

---
## Section 13: Forecast Function â€” User Guide

Call `forecast_weekly_sales()` with any Store/Department inputs to generate a sales forecast. This is the user-facing interface of the AI software artifact.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 13: Forecast Function
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def forecast_weekly_sales(
    store, dept, date, is_holiday,
    temperature, fuel_price, cpi, unemployment,
    store_type_encoded, store_size
):
    """
    Forecast weekly sales for a given Store/Department.

    Parameters
    ----------
    store              : int   Store ID (1â€“45)
    dept               : int   Department ID (1â€“99)
    date               : str   Week date 'YYYY-MM-DD'
    is_holiday         : int   1 = holiday week, 0 = otherwise
    temperature        : float Regional avg temperature (Â°F)
    fuel_price         : float Regional fuel price ($/gallon)
    cpi                : float Consumer Price Index
    unemployment       : float Unemployment rate (%)
    store_type_encoded : int   Store type encoded (A=0, B=1, C=2)
    store_size         : int   Store size in sq ft (e.g. Store 1 = 151315)

    Returns
    -------
    float  Forecasted Weekly Sales ($)
    """
    d = pd.to_datetime(date)
    row = pd.DataFrame([{
        'Store': store, 'Dept': dept,
        'year': d.year, 'month': d.month, 'week': d.isocalendar()[1],
        'IsHoliday': is_holiday, 'Type': store_type_encoded, 'Size': store_size,
        'Temperature': temperature, 'Fuel_Price': fuel_price,
        'CPI': cpi, 'Unemployment': unemployment
    }])
    return max(0.0, float(xgb_model.predict(row[FEATURES_A])[0]))


# â”€â”€ Example predictions â”€â”€
print("=" * 55)
print("Forecast Examples")
print("=" * 55)

# Store sizes from stores.csv: Store 1 = 151315 sq ft (Type A), Store 20 = 203742 sq ft (Type A)
examples = [
    (1,  1,  '2012-10-05', 0, 62.5, 3.52, 211.0, 8.1, 0, 151315),   # Regular week, Store 1
    (1,  1,  '2012-11-23', 1, 45.2, 3.47, 212.5, 8.1, 0, 151315),   # Thanksgiving, Store 1
    (20, 72, '2012-12-28', 1, 38.1, 3.29, 210.2, 7.9, 0, 203742),   # Christmas, Dept 72 (Electronics), Store 20
]

for i, inp in enumerate(examples, 1):
    result = forecast_weekly_sales(*inp)
    tag    = 'Holiday' if inp[3] == 1 else 'Non-Holiday'
    print(f"\nExample {i}: Store {inp[0]}, Dept {inp[1]}, {inp[2]} ({tag})")
    print(f"  Store Size         : {inp[9]:,} sq ft")
    print(f"  Forecasted Sales   : ${result:,.2f}")

---
## Section 14: Final Summary

| Aspect | Detail |
|--------|--------|
| **Strengths** | Handles missing MarkDowns natively; regularization prevents overfitting; time-based split is leak-free; store/dept heterogeneity captured |
| **Limitations** | Cannot extrapolate beyond training range; MarkDowns excluded from primary pipeline |
| **Potential Improvements** | Add lag features; Bayesian hyperparameter tuning; ensemble XGBoost + RF predictions |

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Section 14: Final Results Summary
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"Dataset       : Walmart Store Sales Forecasting")
print(f"Total records : {len(df):,}")
print(f"Train period  : {df.loc[train_mask,'Date'].min().date()} â†’ {df.loc[train_mask,'Date'].max().date()}")
print(f"Test period   : {df.loc[test_mask,'Date'].min().date()}  â†’ {df.loc[test_mask,'Date'].max().date()}")
print(f"Environment   : {'Google Colab' if ON_COLAB else 'Local'}")
print()
print("Primary Model â€” XGBoost Regressor (Scenario A)")
print(f"  Features : {len(FEATURES_A)} | MAE: ${mae_xgb_A:,.2f} | RMSE: ${rmse_xgb_A:,.2f} | WMAPE: {wmape_xgb_A:.2f}%")
print()
print("Validation â€” Random Forest (Scenario A)")
print(f"  Features : {len(FEATURES_A)} | MAE: ${mae_rf_A:,.2f} | RMSE: ${rmse_rf_A:,.2f} | WMAPE: {wmape_rf_A:.2f}%")
print()
print("Secondary â€” XGBoost Scenario B (MarkDowns)")
print(f"  Features : {len(FEATURES_B)} | MAE: ${mae_xgb_B:,.2f} | RMSE: ${rmse_xgb_B:,.2f} | WMAPE: {wmape_xgb_B:.2f}%")
print()
print("Note: WMAPE used in place of MAPE â€” scale-weighted to avoid inflation")
print("      from low-volume rows (~6.4% of dataset has Weekly_Sales < $100).")
print()
print("All plots saved to working directory.")
print("=" * 60)

---
## Section 15: Export Trained Models

Saves the three trained models to disk and downloads them (Colab only).
Place the downloaded files in the same folder as  before running the Streamlit app.
This ensures the GUI shows identical results to this notebook.

In [ ]:
# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─
# Section 15: Export Trained Models
# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─# ─

import joblib
import numpy as np

print(f"XGBoost version used: {xgb.__version__}")

# Save XGBoost Scenario A (primary model)
xgb_model.save_model('xgb_model_A.json')
print("Saved: xgb_model_A.json")

# Save XGBoost Scenario B
xgb_model_B.save_model('xgb_model_B.json')
print("Saved: xgb_model_B.json")

# Save only RF feature importances (avoids 78 MB pkl file for deployment)
np.save('rf_importances.npy', rf_model.feature_importances_)
print("Saved: rf_importances.npy")

# Save test predictions and metadata
np.save('y_test.npy',        y_test.values if hasattr(y_test, "values") else y_test)
np.save('y_pred_xgb_A.npy',  y_pred_xgb_A)
np.save('y_pred_rf_A.npy',   y_pred_rf_A)
np.save('y_pred_xgb_B.npy',  y_pred_xgb_B)
np.save('test_holiday.npy',  test_holiday.values if hasattr(test_holiday, "values") else test_holiday)
test_dates.reset_index(drop=True).to_csv('test_dates.csv', index=False)
print("Saved: test arrays and test_dates.csv")

# Download all files (Colab only)
if ON_COLAB:
    from google.colab import files
    for fname in ['xgb_model_A.json', 'xgb_model_B.json', 'rf_importances.npy',
                  'y_test.npy', 'y_pred_xgb_A.npy', 'y_pred_rf_A.npy',
                  'y_pred_xgb_B.npy', 'test_holiday.npy', 'test_dates.csv']:
        files.download(fname)
        print(f"Downloading: {fname}")
else:
    print("Local run — files saved to:", os.getcwd())

print("
All exports complete. Place these files in the same folder as app.py.")
